# Détection de DDoS avec LightGBM - Pipeline Complète pour Mini-SOC
Ce notebook présente la création d'un modèle d'apprentissage automatique basé sur l'algorithme **LightGBM** pour la détection d'attaques DDoS (spécifiquement LOIC-HTTP) à partir de captures de trafic réseau (CICFlowMeter).

Le pipeline est optimisé pour :
1. **Éviter le surapprentissage (overfitting)** sur les données de laboratoire.
2. **Garantir une bonne généralisation** pour un trafic réseau réel (SOC / Wazuh).
3. **Optimiser les performances** en termes de temps d'entraînement et d'utilisation mémoire.


In [ ]:
# Étape 1 : Installation des dépendances nécessaires
!pip install pandas numpy scikit-learn matplotlib seaborn lightgbm joblib


In [ ]:
# Étape 2 : Importation des bibliothèques
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
import joblib
import json

print('✅ Toutes les bibliothèques sont importées avec succès !')


### Étape 3 : Chargement et Échantillonnage du Dataset
Le dataset complet (`final_dataset.csv`) fait plus de 6 Go. Charger l'intégralité du fichier en mémoire peut saturer la RAM d'une machine standard et provoquer une erreur Out-Of-Memory (OOM). 
Pour ce notebook, nous allons :
- Charger un échantillon de 500 000 lignes pour l'entraînement et l'évaluation.
- Utiliser un état aléatoire (`random_state=42`) pour assurer la reproductibilité.


In [ ]:
# Configuration des chemins et paramètres
FILE_PATH = 'C:/Users/ferye/Desktop/stage/final_dataset_sample.csv'
RANDOM_STATE = 42

print(f'⏳ Chargement du dataset échantillon : {FILE_PATH}...')
try:
    df = pd.read_csv(FILE_PATH, low_memory=False)
    print(f'✅ Dataset chargé avec succès !')
    print(f'Dimensions : {df.shape}')
except FileNotFoundError:
    print(f"❌ Erreur : Le fichier '{FILE_PATH}' n'a pas été trouvé. Veuillez ajuster le chemin.")


### Étape 4 : Analyse Exploratoire Initiale (EDA)
Vérification rapide de la structure du dataset, de la répartition des classes (Benign vs DDoS), et présence de valeurs manquantes ou infinies.


In [ ]:
# Informations générales sur le dataset
print('=' * 50)
print('STRUCTURE DU DATASET')
print('=' * 50)
print(f'Nombre de lignes : {df.shape[0]:,}')
print(f'Nombre de colonnes : {df.shape[1]}')
print(f'Utilisation mémoire : {df.memory_usage(deep=True).sum() / 1024**2:.1f} MB')

# Distribution de la variable cible
label_counts = df['Label'].value_counts()
label_pct = df['Label'].value_counts(normalize=True) * 100

print('\n📊 DISTRIBUTION DES LABELS')
print('-' * 30)
for label in label_counts.index:
    print(f'{label:15s}: {label_counts[label]:>8,} ({label_pct[label]:.2f}%)')


In [ ]:
# Visualisation de la distribution des classes
plt.figure(figsize=(8, 5))
sns.barplot(x=label_counts.index, y=label_counts.values, palette='viridis', edgecolor='black')
plt.title('Distribution des Classes (Benign vs DDoS)', fontsize=14, fontweight='bold')
plt.xlabel('Classe')
plt.ylabel('Nombre d\'occurrences')
for i, v in enumerate(label_counts.values):
    plt.text(i, v + (len(df)*0.01), f'{v:,}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()


### Étape 5 : Nettoyage et Prétraitement
1. **Nettoyage des valeurs infinies et NaN** : Le trafic réseau capturé génère parfois des divisions par zéro (par exemple pour les taux d'octets/paquets si la durée du flux est nulle), ce qui produit des valeurs infinies (`inf`).
2. **Suppression des identifiants non pertinents** : Des colonnes comme `Flow ID`, `Src IP`, `Dst IP`, `Timestamp` et `Src Port` peuvent créer un surapprentissage géolocalisé ou temporel. Le modèle doit apprendre le comportement des paquets, pas des adresses IP spécifiques.


In [ ]:
# 1. Gestion des valeurs manquantes et infinies
print('🔧 Traitement des valeurs infinies et manquantes...')
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if 'Label' in numeric_cols:
    numeric_cols.remove('Label')

# Remplacement des infinis par NaN
df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)

# Imputation par la médiane
nan_count = df[numeric_cols].isnull().sum().sum()
print(f'Nombre de valeurs NaN/infinies identifiées : {nan_count}')
for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())
print('✅ Valeurs manquantes et infinies imputées.')

# 2. Suppression des colonnes inutiles
DROP_COLS = ['Unnamed: 0', 'Flow ID', 'Src IP', 'Dst IP', 'Timestamp']
drop_existing = [col for col in DROP_COLS if col in df.columns]
df = df.drop(columns=drop_existing)
print(f'✅ Colonnes d\'identification supprimées : {drop_existing}')


### Étape 6 : Encodage des Variables
- Encodage de la variable cible `Label` (Benign ➔ 0, DDoS ➔ 1).
- Encodage des variables catégorielles restantes.


In [ ]:
# Encodage de la variable cible
print('🏷️ Encodage de la variable cible...')
df['Label'] = df['Label'].str.lower().str.strip()
label_map = {'benign': 0, 'ddos': 1, 'ddos attacks-loic-http': 1} # gère les variations de casse et de texte

# Si certaines valeurs de label ne matchent pas
df['Label'] = df['Label'].map(label_map)
unknown_labels = df['Label'].isnull().sum()
if unknown_labels > 0:
    print(f'⚠️ {unknown_labels} lignes avec des labels inconnus seront supprimées.')
    df = df.dropna(subset=['Label'])

df['Label'] = df['Label'].astype(int)
print('Distribution finale de la cible :')
print(df['Label'].value_counts())

# Encodage des colonnes catégorielles restantes si existantes
cat_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f'\nVariables catégorielles restantes : {cat_cols}')
le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))
print('✅ Encodage terminé.')


### Étape 7 : Feature Selection et Prévention de l'Overfitting
Dans les datasets réseau de laboratoire (comme CIC-IDS2018), certaines caractéristiques liées aux débits ("Rate Features" comme `Flow Byts/s`, `Flow Pkts/s`, `Fwd Pkts/s`, `Bwd Pkts/s`) sont calculées de manière trop parfaite et permettent de séparer les classes à 100%. En production réelle (trafic issu d'un agent comme Wazuh), ces métriques fluctuent énormément et dépendent de la liaison réseau. 
Si nous laissons le modèle utiliser ces features, il va overfitter sur le jeu de données de laboratoire et échouera lamentablement sur du trafic réel.

**Actions :**
1. **Exclure** les caractéristiques de débit trop parfaites.
2. **Créer une caractéristique de Timestamp** s'il y a lieu, ou utiliser le Feature Importance.
3. **Identifier** les fonctionnalités les plus importantes en utilisant un modèle rapide.
4. **Ajouter un léger bruit gaussien** aux fonctionnalités sélectionnées pour casser la "perfection" artificielle du laboratoire et forcer le modèle à généraliser.


In [ ]:
# Séparation X et y
X = df.drop(columns=['Label'])
y = df['Label']

# 1. Exclusion des features à haut risque d'overfitting
EXCLUDE_RATE_FEATURES = [
    'Flow Byts/s', 
    'Flow Pkts/s', 
    'Fwd Pkts/s', 
    'Bwd Pkts/s'
]
exclude_existing = [f for f in EXCLUDE_RATE_FEATURES if f in X.columns]
X = X.drop(columns=exclude_existing)
print(f'✅ Fiturs de débit exclues : {exclude_existing}')

# 2. Sélection des Top Fiturs avec une LightGBM de pré-sélection rapide
print('🔍 Sélection des caractéristiques les plus importantes...')
selector = lgb.LGBMClassifier(
    n_estimators=50,
    max_depth=5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1
)
selector.fit(X, y)

importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': selector.feature_importances_
}).sort_values('Importance', ascending=False)

TOP_N = 10
SELECTED_FEATURES = importance_df.head(TOP_N)['Feature'].tolist()
print(f'✅ Les {TOP_N} caractéristiques sélectionnées :')
for i, feat in enumerate(SELECTED_FEATURES):
    print(f'   {i+1}. {feat} (Importance: {importance_df.iloc[i]["Importance"]})')

X_selected = X[SELECTED_FEATURES]


In [ ]:
# Plot des Feature Importances
plt.figure(figsize=(10, 6))
sns.barplot(
    x=importance_df.head(TOP_N)['Importance'],
    y=importance_df.head(TOP_N)['Feature'],
    palette='crest'
)
plt.title(f'Top {TOP_N} Caractéristiques les plus Importantes (LightGBM)', fontsize=14, fontweight='bold')
plt.xlabel('Importance (Nombre de splits)')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()


In [ ]:
# 3. Ajout de bruit gaussien (10% de la déviation standard de chaque feature)
NOISE_LEVEL = 0.10
np.random.seed(RANDOM_STATE)
X_noisy = X_selected.copy()

for col in X_selected.columns:
    std = X_selected[col].std()
    if std > 0:
        noise = np.random.normal(0, NOISE_LEVEL * std, size=len(X_selected))
        X_noisy[col] = X_selected[col] + noise

print(f'✅ Bruit gaussien de {NOISE_LEVEL*100}% de la std appliqué aux caractéristiques.')


### Étape 8 : Division Train / Test
Division stratifiée des données pour s'assurer que le ratio de DDoS et de flux normaux reste identique dans l'ensemble d'entraînement et l'ensemble de test.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_noisy, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print('✅ Division Train/Test complétée.')
print(f'  - Train shape : {X_train.shape}')
print(f'  - Test shape  : {X_test.shape}')
print(f'\nRépartition des classes en Train :\n{y_train.value_counts(normalize=True)*100}')
print(f'\nRépartition des classes en Test :\n{y_test.value_counts(normalize=True)*100}')


### Étape 9 : Entraînement du Modèle LightGBM
LightGBM (Light Gradient Boosting Machine) est un algorithme basé sur les arbres de décision boostés par gradient.
Ses avantages majeurs pour la détection d'intrusions / DDoS :
- **Vitesse d'exécution et efficacité mémoire élevées** (grâce à la croissance des arbres par feuille, *leaf-wise growth*).
- **Prise en charge de données massives** sans surcharge de la mémoire.
- **Gestion des classes déséquilibrées** via l'hyperparamètre `scale_pos_weight` ou `class_weight='balanced'`.

Pour éviter le surapprentissage :
- Nous limitons `max_depth` (profondeur max des arbres).
- Nous limitons `num_leaves` (nombre maximal de feuilles par arbre).
- Nous définissons un grand `min_child_samples` (nombre minimal de données dans une feuille).


In [ ]:
print('🚀 Entraînement du modèle LightGBM...')

model = lgb.LGBMClassifier(
    n_estimators=150,
    learning_rate=0.05,
    num_leaves=31,
    max_depth=6,
    min_child_samples=50,
    class_weight='balanced', # Gère le déséquilibre de classe
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[lgb.early_stopping(stopping_rounds=15, verbose=False)]
)

print('✅ Modèle LightGBM entraîné avec succès !')


### Étape 10 : Évaluation du Modèle et Analyse Cybersécurité
Pour un SOC, les métriques classiques ne suffisent pas :
- **Recall (Taux de Détection)** : Doit être extrêmement proche de 100% car rater une attaque DDoS est catastrophique.
- **Precision** : Doit être élevée pour limiter la fatigue des analystes SOC.
- **FAR (False Alarm Rate / Taux de Fausse Alerte)** : C'est le ratio de flux sains classés à tort comme DDoS. Dans un SOC, un FAR > 5% génère des milliers d'alertes inutiles par jour, rendant la surveillance inefficace.


In [ ]:
# Prédictions et probabilités
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

# Calcul des métriques
cm = confusion_matrix(y_test, y_pred)
TN, FP, FN, TP = cm.ravel()

FAR = FP / (FP + TN) if (FP + TN) > 0 else 0
Precision = TP / (TP + FP) if (TP + FP) > 0 else 0
Recall = TP / (TP + FN) if (TP + FN) > 0 else 0
F1 = 2 * (Precision * Recall) / (Precision + Recall) if (Precision + Recall) > 0 else 0
Accuracy = (TP + TN) / (TP + TN + FP + FN)
ROC_AUC = roc_auc_score(y_test, y_proba)

print('=' * 60)
print('📊 RAPPORT DE CLASSIFICATION & EVALUATIONS')
print('=' * 60)
print(classification_report(y_test, y_pred, target_names=['Benign (0)', 'DDoS (1)']))

print('=' * 60)
print('METRIQUES DE PERFORMANCE CYBERSÉCURITÉ')
print('=' * 60)
print(f'Confusion Matrix :')
print(f'  - Vrais Négatifs (Benign correct) : {TN:>8,}')
print(f'  - Faux Positifs (Fausse Alerte)  : {FP:>8,}  ← [FAR]')
print(f'  - Faux Négatifs (DDoS raté)      : {FN:>8,}  ← [🚨 DANGER !]')
print(f'  - Vrais Positifs (DDoS détecté)  : {TP:>8,}')
print()
print(f'⚠️  FAR (False Alarm Rate)   : {FAR*100:.4f}%')
print(f'✅ Taux de Détection (Recall): {Recall*100:.4f}%')
print(f'✅ Précision                : {Precision*100:.4f}%')
print(f'✅ F1-Score                 : {F1*100:.4f}%')
print(f'✅ Accuracy                 : {Accuracy*100:.4f}%')
print(f'✅ ROC-AUC                  : {ROC_AUC*100:.4f}%')
print()

# Interprétation du FAR
if FAR < 0.01:
    print("🟢 Évaluation FAR : Excellent (FAR < 1%)")
elif FAR < 0.05:
    print("🟡 Évaluation FAR : Acceptable pour la production (1% <= FAR < 5%)")
else:
    print("🔴 Évaluation FAR : Trop élevé (FAR >= 5%). Ajustez les seuils ou les hyperparamètres.")


In [ ]:
# Visualisation de la Confusion Matrix et Courbe ROC
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Benign', 'DDoS'])
disp.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Matrice de Confusion', fontsize=14, fontweight='bold')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[1].plot(fpr, tpr, color='#2c3e50', lw=2.5, label=f'Courbe ROC (AUC = {ROC_AUC:.4f})')
axes[1].plot([0, 1], [0, 1], color='#7f8c8d', lw=1.5, linestyle='--', label='Classificateur Aléatoire')
axes[1].fill_between(fpr, tpr, alpha=0.1, color='#2c3e50')
axes[1].set_xlabel('False Positive Rate (FAR)', fontsize=12)
axes[1].set_ylabel('True Positive Rate (Recall)', fontsize=12)
axes[1].set_title('Courbe ROC', fontsize=14, fontweight='bold')
axes[1].legend(loc='lower right', fontsize=11)
axes[1].grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('lightgbm_evaluation.png', dpi=150)
plt.show()


In [ ]:
# Tracé des metriques clés
metrics = {
    'Accuracy': Accuracy,
    'Precision': Precision,
    'Recall': Recall,
    'F1-Score': F1,
    'ROC-AUC': ROC_AUC,
    '1 - FAR': 1 - FAR
}

plt.figure(figsize=(10, 5))
colors = ['#1abc9c' if k != '1 - FAR' else '#e74c3c' for k in metrics.keys()]
bars = plt.bar(metrics.keys(), metrics.values(), color=colors, edgecolor='black', width=0.5)

for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 0.01,
             f'{height:.4f}', ha='center', va='bottom', fontweight='bold')

plt.ylim(0, 1.15)
plt.ylabel('Score')
plt.title('Synthèse des Métriques du Modèle LightGBM', fontsize=14, fontweight='bold')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


### Étape 11 : Sauvegarde du Modèle et de ses Métadonnées
Pour que notre API de détection (FastAPI) ou notre Agent SOC puisse utiliser le modèle en production, nous sauvegardons :
1. Le fichier modèle sérialisé (`ddos_lgb_model.pkl`).
2. Un fichier JSON contenant les métadonnées de configuration (fiturs sélectionnées, etc.) pour s'assurer que l'API applique le même prétraitement.


In [ ]:
# Sauvegarde du modèle
MODEL_PATH = 'ddos_lgb_model.pkl'
joblib.dump(model, MODEL_PATH)
print(f'✅ Modèle sauvegardé sous : {MODEL_PATH}')

# Sauvegarde des métadonnées
metadata = {
    'selected_features': SELECTED_FEATURES,
    'excluded_features': EXCLUDE_RATE_FEATURES,
    'noise_level': NOISE_LEVEL,
    'label_map': {'benign': 0, 'ddos': 1},
    'metrics': {
        'FAR': round(float(FAR), 4),
        'Precision': round(float(Precision), 4),
        'Recall': round(float(Recall), 4),
        'F1': round(float(F1), 4),
        'Accuracy': round(float(Accuracy), 4),
        'ROC_AUC': round(float(ROC_AUC), 4)
    },
    'model_params': model.get_params()
}

# Conversion des types non sérialisables en types natifs
for key, value in metadata['model_params'].items():
    if not isinstance(value, (int, float, str, bool, list, dict, type(None))):
        metadata['model_params'][key] = str(value)

with open('model_metadata_lgb.json', 'w') as f:
    json.dump(metadata, f, indent=2)
print('✅ Fichier de métadonnées model_metadata_lgb.json créé avec succès.')


### Étape 12 : Test de Chargement et Inférence (Simulée)
Vérifions que nous pouvons recharger le modèle sauvegardé et effectuer des prédictions sur de nouveaux échantillons.


In [ ]:
# Test de rechargement
print('🧪 Test de chargement du modèle...')
loaded_model = joblib.load(MODEL_PATH)

# Simulation de prédiction sur 5 exemples de test
sample_data = X_test.head(5)
sample_actual = y_test.head(5).values

predictions = loaded_model.predict(sample_data)
proba = loaded_model.predict_proba(sample_data)[:, 1]

print('\nRésultats du test :')
for i, (pred, prob, actual) in enumerate(zip(predictions, proba, sample_actual)):
    status = '✅' if pred == actual else '❌'
    pred_label = 'DDoS' if pred == 1 else 'Benign'
    actual_label = 'DDoS' if actual == 1 else 'Benign'
    print(f'  Échantillon {i+1} : Pred={pred_label} (Confiance: {prob:.4f}) | Réel={actual_label} {status}')

print('\n🎉 Tout fonctionne correctement. Le modèle est prêt à être déployé avec FastAPI !')
